# DIAGNOSTIC: Feature Importance Analysis & Selection
Identify which features actually matter, remove noise, rebuild model

**Strategy:**
1. Train quick XGBoost on all 445 features
2. Extract feature importance rankings
3. Test models with different feature counts (top 50, 100, 150, 200, 250)
4. Find the sweet spot (best CV + test performance)
5. Train final model with only important features

**Why:** Removes noise, improves signal-to-noise ratio, often beats regularization

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
import gc
import warnings
warnings.filterwarnings('ignore')

print("="*70)
print("FEATURE IMPORTANCE ANALYSIS & SELECTION")
print("="*70)

# ============== LOAD DATA ==============
print("\n[1/4] Loading data...")
train_df = pd.read_parquet('/kaggle/input/competitions/short-horizon-return-prediction-challenge-by-i-rage/train.parquet')
test_df = pd.read_parquet('/kaggle/input/competitions/short-horizon-return-prediction-challenge-by-i-rage/test.parquet')

def compress_dtypes(df):
    for col in df.columns:
        if df[col].dtype != 'object':
            c_min, c_max = df[col].min(), df[col].max()
            if str(df[col].dtype)[:3] == 'int':
                if c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
            else:
                if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
    return df

train_df = compress_dtypes(train_df)
test_df = compress_dtypes(test_df)
train_df = train_df.fillna(train_df.median())
test_df = test_df.fillna(test_df.median())

y_train = train_df['TARGET'].values.copy()
test_ids = test_df['ID'].values.copy()

X_train = train_df.drop(['ID', 'TARGET'], axis=1)
X_test = test_df.drop(['ID'], axis=1)

const_cols = list(set(X_train.columns[X_train.nunique() <= 1]) | set(X_test.columns[X_test.nunique() <= 1]))
X_train = X_train.drop(const_cols, axis=1, errors='ignore')
X_test = X_test.drop(const_cols, axis=1, errors='ignore')

common_cols = sorted(list(set(X_train.columns) & set(X_test.columns)))
X_train = X_train[common_cols].copy()
X_test = X_test[common_cols].copy()

del train_df, test_df
gc.collect()
print(f"✓ Train: {X_train.shape}, Test: {X_test.shape}")

# ============== EXTRACT FEATURE IMPORTANCE ==============
print("\n[2/4] Training quick XGBoost to extract feature importance...")

# Train single model on all data to get importance (quick pass)
xgb_params = {
    'objective': 'reg:squarederror',
    'max_depth': 5,
    'learning_rate': 0.05,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'lambda': 1.0,
    'alpha': 0.5,
    'verbosity': 0
}

model_importance = xgb.XGBRegressor(**xgb_params, n_estimators=300, random_state=42)
model_importance.fit(X_train, y_train, verbose=False)

# Get feature importance
importance_dict = model_importance.get_booster().get_score(importance_type='weight')
importance_df = pd.DataFrame(list(importance_dict.items()), columns=['feature', 'importance'])
importance_df = importance_df.sort_values('importance', ascending=False).reset_index(drop=True)

print(f"✓ Feature importance extracted")
print(f"\nTop 20 Important Features:")
print(importance_df.head(20).to_string())
print(f"\n... and {len(importance_df) - 20} more features")

# ============== TEST DIFFERENT FEATURE COUNTS ==============
print("\n[3/4] Testing different feature counts (single KFold, 3 splits for speed)...\n")

feature_counts = [50, 100, 150, 200, 250, 300]
results = []

for feature_count in feature_counts:
    print(f"Testing with TOP {feature_count} features...")
    
    top_features = importance_df.head(feature_count)['feature'].tolist()
    X_train_sel = X_train[top_features]
    X_test_sel = X_test[top_features]
    
    # Quick CV: 3 folds instead of 5
    kf = KFold(n_splits=3, shuffle=True, random_state=42)
    fold_scores = []
    
    for train_idx, val_idx in kf.split(X_train_sel):
        X_tr, X_val = X_train_sel.iloc[train_idx], X_train_sel.iloc[val_idx]
        y_tr, y_val = y_train[train_idx], y_train[val_idx]
        
        model = xgb.XGBRegressor(**xgb_params, n_estimators=300, random_state=42)
        model.fit(X_tr, y_tr, verbose=False)
        
        r2 = r2_score(y_val, model.predict(X_val))
        fold_scores.append(r2)
        
        del model, X_tr, X_val, y_tr, y_val
        gc.collect()
    
    avg_r2 = np.mean(fold_scores)
    results.append({
        'feature_count': feature_count,
        'cv_r2': avg_r2,
        'best_fold': max(fold_scores),
        'worst_fold': min(fold_scores)
    })
    print(f"  → CV R² (3-fold avg): {avg_r2:.6f} (range: {min(fold_scores):.6f} to {max(fold_scores):.6f})")

# Show results
results_df = pd.DataFrame(results)
best_idx = results_df['cv_r2'].idxmax()
best_count = results_df.loc[best_idx, 'feature_count']
best_r2 = results_df.loc[best_idx, 'cv_r2']

print(f"\n" + "="*70)
print(f"FEATURE COUNT ANALYSIS:")
print(f"="*70)
print(results_df.to_string(index=False))
print(f"\n✓ BEST: {int(best_count)} features → CV R² = {best_r2:.6f}")

del model_importance
gc.collect()

## Results Analysis
The table above shows how CV R² changes with different feature counts. Look for:
- **Peak R²**: Best validation score
- **Stability**: Smaller gap between best/worst fold = more stable

Next cell will use the best feature count for final submission.

In [ ]:
# ============== FINAL MODEL WITH BEST FEATURES ==============
print("\n[4/4] Training final ensemble with BEST FEATURES (3 seeds)...\n")

# Verify that best_count is defined from cell 2
if 'best_count' not in locals():
    print("ERROR: best_count not defined. Please run cell 2 first!")
else:
    # Use the best feature count identified above
    top_features = importance_df.head(int(best_count))['feature'].tolist()
    X_train_final = X_train[top_features]
    X_test_final = X_test[top_features]
    
    print(f"Using {len(top_features)} features:")
    if len(top_features) > 10:
        print(top_features[:10], "...")
    else:
        print(top_features)
    
    seeds = [42, 123, 456]
    ensemble_preds = np.zeros(len(X_test_final))
    all_scores = []
    
    # Optimized hyperparameters (moderate, not extreme)
    xgb_params_final = {
        'objective': 'reg:squarederror',
        'max_depth': 5,              # Moderate depth
        'learning_rate': 0.05,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'lambda': 2.0,               # Moderate regularization
        'alpha': 1.0,
        'verbosity': 0
    }
    
    for seed_idx, seed in enumerate(seeds, 1):
        print(f"  Model {seed_idx}/3 (seed={seed}):")
        
        kf = KFold(n_splits=5, shuffle=True, random_state=seed)
        fold_indices = list(kf.split(X_train_final))
        
        test_preds = np.zeros(len(X_test_final))
        fold_scores = []
        
        for fold_num, (train_idx, val_idx) in enumerate(fold_indices, 1):
            X_tr, X_val = X_train_final.iloc[train_idx], X_train_final.iloc[val_idx]
            y_tr, y_val = y_train[train_idx], y_train[val_idx]
            
            model = xgb.XGBRegressor(**xgb_params_final, n_estimators=400, random_state=seed)
            model.fit(X_tr, y_tr, verbose=False)
            
            test_preds += model.predict(X_test_final) / 5
            r2 = r2_score(y_val, model.predict(X_val))
            fold_scores.append(r2)
            
            del model, X_tr, X_val, y_tr, y_val
            gc.collect()
        
        avg_r2 = np.mean(fold_scores)
        all_scores.append(avg_r2)
        ensemble_preds += test_preds / 3
        print(f"    Avg R² (5-fold): {avg_r2:.6f}")
    
    print(f"\n✓ Final ensemble trained (3 models averaged)")
    print(f"✓ Individual model scores: {[f'{s:.6f}' for s in all_scores]}")
    print(f"✓ Ensemble Avg R²: {np.mean(all_scores):.6f}")
    
    # ============== CREATE SUBMISSION ==============
    print(f"\n[5/5] Creating submission...")
    submission = pd.DataFrame({'ID': test_ids, 'TARGET': ensemble_preds})
    print(f"✓ Submission shape: {submission.shape}")
    print(f"✓ Target - Mean: {submission['TARGET'].mean():.6f}, Std: {submission['TARGET'].std():.6f}")
    print(f"✓ Target - Min: {submission['TARGET'].min():.6f}, Max: {submission['TARGET'].max():.6f}")
    
    submission.to_csv('submission_feature_selection.csv', index=False)
    print(f"✓ Saved: submission_feature_selection.csv")
    
    print("\n" + "="*70)
    print("FEATURE SELECTION - READY FOR KAGGLE")
    print("="*70)
    print(f"✓ Strategy: Use only {int(best_count)} most important features")
    print(f"✓ Removed noise from ~445 original features")
    print(f"✓ Moderate hyperparameters (depth=5, λ=2, α=1)")
    print(f"✓ Ensemble: 3 seeds with 5-fold CV each")
    print("="*70)